# 06 — Same-traffic-type neighbours (experiment)
Notebooks 02–04 showed that the **X closest users add no predictive value** over the X=0 baseline
(demand-driven throughput, no contention signal). This notebook tests a sharper neighbourhood definition:
the X nearest users **running the same traffic class at the same instant**. Hypothesis: co-located users on
the same service may have *correlated demand*, making them more informative than generic neighbours.

Design: identical pipeline to 02/03 (same population, split seed, outlier rule, aggregated encoding) —
the **only** experimental variable is the neighbour pool. Results append to `results/metrics_sametype.csv`
and are compared against `results/metrics.csv` (X=0 baseline and generic-pool `agg`).

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); keep >= 10
                                 # so the nearest-alignment tolerance stays above the worst raw gap.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
SAMETYPE_X       = [3, 5]        # X values for THIS experiment (agg encoding only: notebook 04 showed
                                 # the positional encoding only adds permutation noise)
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): at full population the top ~1% of active samples carry
                                 # ~86% of the total variance -> MSE/R2 would be about a handful of rare peaks.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw "wide" format -> tidy per-user/per-timestamp table) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    """User-id range covered by a CSV chunk, taken from its file name (…_<start>_<end>.csv)."""
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    """CSV chunks of a metric, keeping only files whose user-id range intersects `user_ids`."""
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(f):
            a, b = file_id_range(f)
            return any(a <= u <= b for u in user_ids)
        files = [f for f in files if overlaps(f)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    """Every user id present in the venue (derived from the throughput file names)."""
    ids = []
    for f in metric_files(venue_dir, "*Throughput*", "*.csv"):
        a, b = file_id_range(f)
        ids.extend(range(a, b + 1))
    return np.array(sorted(ids))

def build_grid(ref_file, resample_seconds):
    t = pd.read_csv(ref_file, usecols=[0]).iloc[:, 0].values.astype(float)
    return np.arange(t[0], t[-1] + 1, resample_seconds)

def _align(df, grid, tol):
    df = df[~df.index.duplicated(keep="first")]   # raw stamps contain duplicates (diff = 0s)
    return df.reindex(grid, method="nearest", tolerance=tol)

def load_metric(files, value_name, grid, tol, user_ids):
    out = []
    for f in files:
        head = list(pd.read_csv(f, nrows=0).columns)
        cmap = {c: int(re.search(r'(\d+)', c).group(1)) for c in head[1:]}
        use = [head[0]] + [c for c in head[1:] if cmap[c] in user_ids]   # parse only sampled users
        df = pd.read_csv(f, usecols=use).rename(columns={head[0]: "time"}).set_index("time")
        df = df.rename(columns=cmap)
        df = _align(df, grid, tol).astype("float32")   # float32 halves RAM (full population fits Colab)
        out.append(df.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, grid, tol, user_ids):
    """Positions are interleaved blocks of (id, lat, lon, alt, mobileState); mobileState is the
    traffic_type. lat/lon are converted to local metres using ONE shared origin for the whole venue,
    so distances between users coming from different files are consistent."""
    frames = []
    for f in files:
        first = pd.read_csv(f, nrows=1).values.astype(float)
        ids_all = first[0, 1::5].astype(int)
        blocks = [k for k, u in enumerate(ids_all) if u in user_ids]
        if not blocks:
            continue
        use = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]    # parse only sampled users
        df = pd.read_csv(f, usecols=use)
        t = df.iloc[:, 0].values.astype(float)
        arr = df.values.astype(float)
        ids = arr[0, 1::5].astype(int)
        vals = {"lat": arr[:, 2::5], "lon": arr[:, 3::5], "z": arr[:, 4::5], "traffic_type": arr[:, 5::5]}
        m = None
        for name, v in vals.items():
            d = pd.DataFrame(v, index=t, columns=ids)
            d.index.name = "time"
            d = _align(d, grid, tol).reset_index().melt(id_vars="time", var_name="user_id", value_name=name)
            m = d if m is None else m.merge(d, on=["time", "user_id"])
        frames.append(m)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()          # single origin for the whole venue
    R = 6371000.0
    pos["x"] = R * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = R * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    """Return a tidy DataFrame with one row per (user, timestamp).
    n_users=None uses the venue's FULL population (recommended: the X-closest neighbourhoods are the
    real ones). random_users=True draws n_users uniformly from the full population (seeded,
    reproducible); otherwise the first n_users ids are used."""
    venue = find_venue_dir(DATA_ROOT, venue_key)
    pop = all_user_ids(venue)
    if n_users is None:
        user_ids = set(int(u) for u in pop)
        print(f"{venue_key}: using ALL {len(user_ids)} users")
    elif random_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(int(u) for u in rng.choice(pop, size=min(n_users, len(pop)), replace=False))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(pop)}")
    else:
        user_ids = set(range(n_users))
    # nearest-alignment tolerance: half the grid step, but never below the raw data's worst gap.
    # Raw stamps are jittered (ACC: nominal 3s but ~1/3 of steps are 4s, plus duplicates and holes
    # up to 7s), so a floor of 4s guarantees every grid point finds its sample even at fine grids.
    tol = max(resample_seconds / 2, 4.0)
    mf = lambda sub, fg: metric_files(venue, sub, fg, user_ids)
    ref = mf("*Throughput*", "*.csv")[0]
    grid = build_grid(ref, resample_seconds)
    parts = [
        load_metric(mf("*Throughput*",     "*.csv"),    "throughput", grid, tol, user_ids),
        load_metric(mf("*BLER*",           "*.csv"),    "bler",       grid, tol, user_ids),
        load_metric(mf("*PRB*",            "*.csv"),    "prb",        grid, tol, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"),    "ru_id",      grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRDL_*.csv"), "sinr_dl", grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRUL_*.csv"), "sinr_ul", grid, tol, user_ids),
        load_positions(mf("*Positions*",   "*.csv"), grid, tol, user_ids),
    ]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p, on=["time", "user_id"], how="inner")
    df = df.dropna().reset_index(drop=True)
    df["user_id"]      = df["user_id"].astype(int)
    df["traffic_type"] = df["traffic_type"].round().astype(int)
    df["ru_id"]        = df["ru_id"].round().astype(int)
    return df

In [ ]:
# === Same-traffic-type closest-users features ===
# Same vectorised KDTree engineering as notebook 02, with ONE change: candidates are grouped by
# (time, traffic_type) instead of (time,) — each user's X nearest neighbours are searched only among
# users running the SAME traffic class at that instant. Hypothesis under test: co-located users on the
# same service may have correlated demand, making them more informative than generic neighbours.
# Computed BEFORE the ACTIVE_ONLY filter (as in 02); an active user's same-type neighbours are
# necessarily active-class users, so the pool change is the only difference vs notebook 02.
NEIGHBOR_FEATS = ["sinr_dl", "sinr_ul", "prb", "bler"]

def add_same_type_neighbor_features(df, x_max):
    cols = []
    for k in range(x_max):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    out = np.full((len(df), len(cols)), np.nan, dtype="float32")
    feat = df[NEIGHBOR_FEATS].values
    pos = df[["x", "y", "z"]].values
    for _, idx in df.groupby(["time", "traffic_type"]).groups.items():
        idx = np.asarray(idx)
        n = len(idx)
        if n <= 1:
            continue
        pts = pos[idx]
        k = min(x_max + 1, n)
        dist, nb = cKDTree(pts).query(pts, k=k)
        rows = np.arange(n)[:, None]
        not_self = nb != rows                                 # self appears exactly once per row
        order = np.argsort(~not_self, axis=1, kind="stable")  # non-self first, distance order kept
        take = min(x_max, k - 1)
        r = np.repeat(np.arange(n), take)
        c = order[:, :take].ravel()
        block = np.empty((n, take, 1 + len(NEIGHBOR_FEATS)), dtype="float32")
        block[:, :, 0] = dist[r, c].reshape(n, take)
        block[:, :, 1:] = feat[idx[nb[r, c]]].reshape(n, take, -1)
        out[idx, :take * (1 + len(NEIGHBOR_FEATS))] = block.reshape(n, -1)
    return pd.concat([df, pd.DataFrame(out, columns=cols, index=df.index)], axis=1)

# order-invariant aggregates — IDENTICAL semantics to notebook 02, so the only experimental
# variable is the neighbour pool (all users vs same traffic type)
AGG_FEATS = ["nb_prb_sum", "nb_throughput_sum", "nb_sinr_dl_mean", "nb_sinr_ul_mean",
             "nb_bler_mean", "nb_active_count"]

def aggregate_neighbor_features(frame, x):
    def block(feat):
        return frame[[f"nb{k}_{feat}" for k in range(x)]]
    prb = block("prb")
    return pd.DataFrame({
        "nb_prb_sum":        prb.sum(axis=1, min_count=1),
        "nb_throughput_sum": block("throughput").sum(axis=1, min_count=1),
        "nb_sinr_dl_mean":   block("sinr_dl").mean(axis=1),
        "nb_sinr_ul_mean":   block("sinr_ul").mean(axis=1),
        "nb_bler_mean":      block("bler").mean(axis=1),
        "nb_active_count":   (prb.fillna(0) > 0).sum(axis=1).astype(float),
    }, index=frame.index)

## Build the table with same-type neighbours
Neighbour features are computed on the full population **before** `ACTIVE_ONLY` (as in 02). Grouping is by
`(time, traffic_type)`: a video user's neighbours are the nearest *video* users at that instant. Users whose
class has no other member at that instant get NaN neighbours (→ train-median fill), so we also report how
often a same-type neighbour exists at all.

In [ ]:
import time as _time
t0 = _time.time()
df = assemble("acc", n_users=N_USERS, resample_seconds=RESAMPLE_SECONDS, random_users=True)
print("tidy shape:", df.shape, "| users:", df.user_id.nunique(), "| timestamps:", df.time.nunique())
df = add_same_type_neighbor_features(df, x_max=max(SAMETYPE_X))
print("with same-type neighbour features:", df.shape, "| built in %.1fs" % (_time.time()-t0))
# how often does a same-type neighbour even exist? (small groups -> NaN -> median fill)
print("rows with at least 1 same-type neighbour: %.1f%%" % (df.nb0_dist.notna().mean()*100))

if ACTIVE_ONLY:
    before = len(df)
    df = df[df.traffic_type >= MIN_TRAFFIC_TYPE].reset_index(drop=True)
    print(f"ACTIVE_ONLY: kept {len(df)}/{before} rows (traffic_type >= {MIN_TRAFFIC_TYPE})")

In [ ]:
# same seed and procedure as notebook 02 -> identical user split and outlier rule, so metrics are
# directly comparable with results/metrics.csv
rng = np.random.default_rng(RANDOM_SEED)
users = df.user_id.unique(); rng.shuffle(users)
n = len(users); tr = users[:int(.7*n)]; va = users[int(.7*n):int(.85*n)]; te = users[int(.85*n):]
split = {u:"train" for u in tr}; split.update({u:"val" for u in va}); split.update({u:"test" for u in te})
df["split"] = df.user_id.map(split)

if OUTLIER_PCT is not None:
    thr = float(np.percentile(df.loc[df.split=="train", "throughput"], OUTLIER_PCT))
    before = len(df)
    df = df[df.throughput <= thr].reset_index(drop=True)
    print(f"outlier threshold (p{OUTLIER_PCT} of train) = {thr:.2f} Mbps -> removed {before-len(df)} rows")
print(df.split.value_counts())

## Feature matrices (aggregated encoding only)
Notebook 04 showed the positional encoding only adds permutation noise under co-location, so this experiment
uses the **aggregated** encoding: same `AGG_FEATS` as 02, identical semantics — only the pool changes.

In [ ]:
from sklearn.preprocessing import StandardScaler
import json, joblib

BASE_NUM = ["bler","prb","sinr_dl","sinr_ul","x","y","z"]
TRAFFIC_CLASSES = [0,1,2,3,4,5]

def onehot_traffic(frame):
    return pd.DataFrame({f"traffic_{c}": (frame.traffic_type==c).astype(float) for c in TRAFFIC_CLASSES},
                        index=frame.index)

def build_matrix_agg(frame, x, scaler=None, fit=False):
    frame = pd.concat([frame, aggregate_neighbor_features(frame, x)], axis=1)
    num_cols = BASE_NUM + AGG_FEATS
    num = frame[num_cols].astype("float32")
    num = num.fillna(num.median())            # tiny same-type groups -> column median
    if fit:
        scaler = StandardScaler().fit(num)
    num_s = pd.DataFrame(scaler.transform(num), columns=num_cols, index=frame.index)
    X = pd.concat([num_s, onehot_traffic(frame)], axis=1)
    return X.values.astype("float32"), list(X.columns), scaler

tr_m, va_m, te_m = df.split=="train", df.split=="val", df.split=="test"
y = df["throughput"].values.astype("float32")
data = {}
for x in SAMETYPE_X:
    Xtr, cols, scaler = build_matrix_agg(df[tr_m], x, fit=True)
    Xva, _, _ = build_matrix_agg(df[va_m], x, scaler=scaler)
    Xte, _, _ = build_matrix_agg(df[te_m], x, scaler=scaler)
    data[x] = (Xtr, y[tr_m], Xva, y[va_m], Xte, y[te_m])
    joblib.dump(scaler, f"{PROCESSED_DIR}/acc_X{x}_same_scaler.pkl")
    json.dump(cols, open(f"{PROCESSED_DIR}/acc_X{x}_same_cols.json","w"))
    print(f"X={x}  features={Xtr.shape[1]}  train={Xtr.shape[0]}")

## Train NN + RF (same models and tuning as notebook 03)

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    return {"MSE": float(mean_squared_error(y_true, y_pred)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2":  float(r2_score(y_true, y_pred))}

In [ ]:
import json, time, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
tf.random.set_seed(RANDOM_SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
def build_mlp(input_dim, units=64, lr=1e-3, depth=2):
    m = keras.Sequential([keras.layers.Input((input_dim,))])
    for _ in range(depth):
        m.add(keras.layers.Dense(units, activation="relu"))
    m.add(keras.layers.Dense(1))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=["mae"])
    return m

def train_nn(Xtr, ytr, Xva, yva):
    best, best_val, best_cfg = None, np.inf, None
    es = keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
    for units in (64, 128):                       # small validation-based search (GPU: a few seconds each)
        m = build_mlp(Xtr.shape[1], units=units, lr=1e-3)
        m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=1024,
              callbacks=[es], verbose=0)
        v = m.evaluate(Xva, yva, verbose=0)[0]
        if v < best_val:
            best, best_val, best_cfg = m, v, {"units": units, "lr": 1e-3}
    return best, best_cfg

In [ ]:
RF_TUNE_SUBSAMPLE = 20000   # rows used for the CV search (final model is refit on ALL training rows)

def train_rf(Xtr, ytr):
    if len(Xtr) > RF_TUNE_SUBSAMPLE:
        s = np.random.RandomState(RANDOM_SEED).choice(len(Xtr), RF_TUNE_SUBSAMPLE, replace=False)
        Xt, yt = Xtr[s], ytr[s]
    else:
        Xt, yt = Xtr, ytr
    base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                                 max_features="sqrt", min_samples_leaf=2, max_samples=0.3)
    grid = {"n_estimators": [100, 200], "max_depth": [16, None]}
    gs = GridSearchCV(base, grid, cv=3, scoring="neg_mean_squared_error", n_jobs=1)
    gs.fit(Xt, yt)
    best = gs.best_estimator_
    best.fit(Xtr, ytr)                      # refit best config on the full training set
    return best, gs.best_params_

In [ ]:
def infer_ms(predict, X):
    t = time.time(); predict(X); return round((time.time()-t)/len(X)*1000, 4)

results = []
for x in SAMETYPE_X:
    Xtr, ytr, Xva, yva, Xte, yte = data[x]

    t = time.time(); nn, nn_cfg = train_nn(Xtr, ytr, Xva, yva); nn_dur = time.time()-t
    nn_m = evaluate(yte, nn.predict(Xte, verbose=0).ravel())
    nn_m.update(model="NN", X=x, enc="agg_same", train_s=round(nn_dur,1),
        infer_ms=infer_ms(lambda X: nn.predict(X, verbose=0), Xte), cfg=str(nn_cfg))
    nn.save(f"{RESULTS_DIR}/models/nn_X{x}_same.keras")

    t = time.time(); rf, rf_cfg = train_rf(Xtr, ytr); rf_dur = time.time()-t
    rf_m = evaluate(yte, rf.predict(Xte))
    rf_m.update(model="RF", X=x, enc="agg_same", train_s=round(rf_dur,1),
        infer_ms=infer_ms(rf.predict, Xte), cfg=str(rf_cfg))
    joblib.dump(rf, f"{RESULTS_DIR}/models/rf_X{x}_same.pkl")

    results += [nn_m, rf_m]
    print(f"X={x} same | NN R2={nn_m['R2']:.3f} ({nn_dur:.0f}s) | RF R2={rf_m['R2']:.3f} ({rf_dur:.0f}s)")

res_same = pd.DataFrame(results)
res_same.to_csv(f"{RESULTS_DIR}/metrics_sametype.csv", index=False)
res_same

## Comparison: baseline vs generic neighbours vs same-type neighbours

In [ ]:
import matplotlib.pyplot as plt
GREEN, RED, INK, MUT = "#2a9d8f", "#e76f51", "#333333", "#777777"   # project palette

main = pd.read_csv(f"{RESULTS_DIR}/metrics.csv")
if "enc" not in main.columns:
    main["enc"] = "pos"
scen = [("X=0\nbaseline", main[main.enc == "none"])]
scen += [(f"X={x}\nagg (all)", main[(main.enc == "agg") & (main.X == x)]) for x in SAMETYPE_X]
scen += [(f"X={x}\nagg (same type)", res_same[res_same.X == x]) for x in SAMETYPE_X]
scen = [(name, d) for name, d in scen if len(d)]          # tolerate a partially-filled metrics.csv

fig, ax = plt.subplots(figsize=(8.5, 4.4))
w, xs = 0.35, np.arange(len(scen))
for off, model, color in [(-w/2, "NN", GREEN), (w/2, "RF", RED)]:
    vals = [float(d.loc[d.model == model, "R2"].iloc[0]) if (d.model == model).any() else np.nan
            for _, d in scen]
    bars = ax.bar(xs + off, vals, width=w, color=color, label=model)
    for b, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width()/2, v + .004, f"{v:.3f}", ha="center", color=INK, fontsize=9)
base_rf = scen[0][1]
if (base_rf.model == "RF").any():
    ax.axhline(float(base_rf.loc[base_rf.model == "RF", "R2"].iloc[0]), color=MUT, lw=1, ls=":")
ax.set_xticks(xs, [n for n, _ in scen])
ax.set_ylabel("R\u00b2 (test)"); ax.set_ylim(0, max(.42, ax.get_ylim()[1]))
ax.set_title("Do same-traffic-type neighbours beat generic neighbours?")
ax.legend(frameon=False, labelcolor=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/06_sametype_comparison.png", dpi=120); plt.show()

In [ ]:
# If same-type neighbours DO move the needle, check WHERE the signal comes from before celebrating:
# a high nb_throughput_sum importance would mean co-located same-type users share their (test) labels
# through the feature — a channel to report carefully, not a modelling win.
import json as _json
x_imp = SAMETYPE_X[0]
rf = joblib.load(f"{RESULTS_DIR}/models/rf_X{x_imp}_same.pkl")
cols = _json.load(open(f"{PROCESSED_DIR}/acc_X{x_imp}_same_cols.json"))
imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
top = imp.head(12)[::-1]
plt.figure(figsize=(7, 4.2))
top.plot.barh(color=[RED if c.startswith("nb") else GREEN for c in top.index])
plt.title(f"RF importance, X={x_imp} same-type (green = own, red = neighbours)")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/06_sametype_importance.png", dpi=120); plt.show()
print("nb_throughput_sum importance: %.4f" % imp.get("nb_throughput_sum", 0.0))

## Reading the result
- **Same-type ≈ baseline** (expected): confirms that even demand-matched neighbours carry no signal —
  per-user demand is independent and the cell is not contended. The negative result now covers *both*
  neighbourhood definitions, which closes the question properly in the report.
- **Same-type > baseline**: check the importance cell before claiming a win — if `nb_throughput_sum`
  drives it, the gain is co-located same-type users sharing their labels through the feature
  (a channel unavailable in deployment), not a usable predictor.